In [30]:
import os
import numpy as np
import pandas as pd
import networkx as nx
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

# CONFIGURATION
YEARS_TO_PROCESS = [2018] # Adjust list to run specific years
SIMILARITY_THRESHOLD = 0.90 

FOOD_WHITELIST = ["vihannes", "vihannekset", "kasvis", "kasvikset", "juures", "juurekset",
                    "salaatti", "hedelmä", "hedelmät", "marja", "marjat", "pähkinä", "pähkinät",
                    "siemenet", "kuitu", "kuitua", "proteiini", "valkuainen", "vitamiini", "hivenaine",
                    "terveellinen", "terveys", "ruokavalio", "vähärasvainen", "kevyt", "rasvaton", "kotiruoka",
                    "itsetehty", "tuore", "lähiruoka", "peruna", "porkkana", "sipuli", "kala", "lohi", "kana",
                    "broileri", "liha", "naudanliha", "kananmuna", "maito", "pikaruoka", "roskaruoka", "eines",
                    "einekset", "valmisruoka", "sokeri", "sokeria", "makeinen", "karkki", "karkit", "suklaa", "herkku",
                    "herkut", "rasvainen", "epäterveellinen", "lihava", "lihavuus", "lihoaminen", "sipsit", "sipsi", "pizza",
                    "pitsa", "hampurilainen", "purilainen", "kebab", "ranskalaiset", "limsa", "limu", "alkoholi", "olut",
                    "viini", "viina", "siideri", "prosessoitu", "lisäaine", "kasvissyöjä", "kasvisruoka", "vegaani", "vegaaninen",
                    "veg", "kaurajuoma", "soija", "tofu", "härkis", "nyhtökaura", "lihankulutus", "luomu", "ilmastoystävällinen",
                    "hiilijalanjälki", "ilmasto", "päästöt", "ympäristö", "kotimainen", "suomalainen", "tuotanto", "hävikki", "ruokahävikki",
                    "eettinen", "eläinsuojelu", "hinta", "halpa", "kallis", "tarjous", "ale", "euroa", "euro", "kauppa", "ruokakauppa", "market",
                    "lidl", "prisma", "citymarket", "k-kauppa", "s-ryhmä", "budjetti", "opiskelija"]


RELIGIOUS_BLACKLIST = ["jumala", "jumalan", "jumalasta", "jeesus", "jeesuksen", "kristus", "kristuksen", "herra", "herran", "raamattu", "raamatun",
                       "pyhä", "henki", "aamen", "evankeliumi", "synti", "synnin", "armo", "pelastus", "seurakunta", "usko", "uskonto", "uskova", "uskovat", "profetta", "opetuslapsi"]

def build_network(year):
    filename = f"suomi24_food_{year}.csv"
    if not os.path.exists(filename): 
        return None
        
    df = pd.read_csv(filename).dropna(subset=['post_content'])
    t_df = df.groupby('thread_id')['post_content'].apply(' '.join).reset_index()
    
    mask = t_df['post_content'].str.contains('|'.join(RELIGIOUS_BLACKLIST), case=False, na=False)
    t_df = t_df[~mask].copy()

    t_df['word_count'] = t_df['post_content'].apply(lambda x: sum(1 for w in FOOD_WHITELIST if w in str(x).lower()))
    t_df = t_df[t_df['word_count'] >= 4].copy()
    
    if len(t_df) < 2: 
        return None

    tfidf_matrix = TfidfVectorizer(vocabulary=FOOD_WHITELIST).fit_transform(t_df['post_content'])
    sim_matrix = cosine_similarity(tfidf_matrix)
    
    G = nx.Graph()
    t_ids = t_df['thread_id'].tolist()
    G.add_nodes_from(t_ids)
    
    rows, cols = np.where(sim_matrix >= SIMILARITY_THRESHOLD)
    G.add_weighted_edges_from([(t_ids[r], t_ids[c], float(sim_matrix[r, c])) for r, c in zip(rows, cols) if r < c])
    G.remove_nodes_from(list(nx.isolates(G)))
    
    nx.write_gexf(G, f"thread_network_{year}.gexf")
    print(f"Year {year} -> Nodes: {G.number_of_nodes()}, Edges: {G.number_of_edges()}")
    return G

for y in YEARS_TO_PROCESS:
    build_network(y)

Year 2018 -> Nodes: 2195, Edges: 39533


In [32]:
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer

TARGET_YEAR = 2018
TEXT_FILE = f"suomi24_food_{TARGET_YEAR}.csv"
GEPHI_FILE = f"nodes{TARGET_YEAR}_0.60.csv" # Ensure your gephi export matches this format

STOPWORDS = ["olla", "on", "ole", "oli", "olen", "olet", "olemme", "olette", "ovat", "olisi", "olleet",
                "ollut", "voida", "voi", "voivat", "voisi", "pitää", "pitäisi", "tulla", "tulee", "tuli",
                "saada", "saa", "saavat", "tehdä", "tekee", "teki", "mennä", "menee", "meni", "sanoa",
                "sanoo", "sanoi", "ja", "ei", "en", "et", "emme", "ette", "eivät", "että", "tai", "vai",
                "vaan", "mutta", "kun", "jos", "vaikka", "kuin", "myös", "niin", "sekä", "siis", "eli",
                "kyllä", "ehkä", "se", "sen", "sitä", "ne", "niiden", "niitä", "tämä", "tämän", "tätä",
                "nämä", "näiden", "näitä", "tuo", "tuon", "tuota", "nuo", "joka", "jonka", "jota", "jotka",
                "joiden", "joita", "mikä", "minkä", "mitä", "mitkä", "kuka", "ketkä", "kenen", "ketä", "minä",
                "minun", "minua", "mä", "mun", "mua", "me", "meidän", "meitä", "sinä", "sinun", "sinua", "sä",
                "sun", "sua", "te", "teidän", "teitä", "hän", "hänen", "häntä", "he", "heidän", "heitä", "itse",
                "itsensä", "tosi", "ihan", "vain", "vielä", "jo", "nyt", "sitten", "taas", "heti", "aina", "joskus",
                "no", "vähän", "paljon", "muutama", "usein", "kanssa", "mukaan", "takia", "vuoksi", "läpi", "asti",
                "täällä", "siellä", "missä", "sinne", "tänne", "oy", "suomen", "suomessa", "keski", "kunta", "kunnat",
                "euroa", "miljoonaa", "eikä", "joku", "koska", "mitään", "siinä", "siitä", "siihen", "alkaa", "kaikki",
                "edes", "noin", "hyvä", "hyvää"]

df_merged = pd.merge(pd.read_csv(TEXT_FILE), pd.read_csv(GEPHI_FILE), left_on='thread_id', right_on='Id').dropna(subset=['post_content'])
top_classes = df_merged['modularity_class'].value_counts().head(8).index

for mod_class in top_classes:
    text = df_merged[df_merged['modularity_class'] == mod_class]['post_content']
    vectorizer = TfidfVectorizer(max_features=10, stop_words=STOPWORDS, max_df=0.6)
    
    try:
        tfidf_matrix = vectorizer.fit_transform(text)
        words = vectorizer.get_feature_names_out()
        top_words = [words[i] for i in tfidf_matrix.sum(axis=0).A1.argsort()[::-1]]
        print(f"Class {mod_class} ({len(text)} posts): {', '.join(top_words)}")
    except ValueError:
        pass

Class 71 (499 posts): alkoholi, alkoholia, alkoholin, kuten, enemmän, sillä, enää, tässä, hyvin, ennen
Class 59 (480 posts): sokeria, vettä, päälle, pois, kg, dl, 10, tl, voita, 1dl
Class 146 (399 posts): lihava, nainen, mies, naiset, ettei, miehen, miten, miehet, kuinka, naisten
Class 160 (307 posts): maksaa, ostaa, 20, yli, 10, 50, 30, alkoholia, eu, rahaa
Class 111 (292 posts): pizza, pizzaa, pizzat, kebab, onko, pizzan, hyvin, todella, pizzeria, vikaa
Class 102 (214 posts): lihavuus, lihava, lihavia, nainen, hyvin, lihavat, syö, toki, enemmän, mies
Class 36 (203 posts): b12, vitamiini, vitamiinin, vitamiinia, anemia, hyvin, oireet, oireita, puutos, ym
Class 72 (202 posts): ruokavalio, aika, enemmän, lihaa, syö, jossa, liian, kannattaa, jne, diabeteksen
